## Config

In [1]:
import os
import csv
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torch.amp import GradScaler, autocast

DATA_DIR = 'data'
VOCAB_PATH = os.path.join(DATA_DIR, 'vocab.txt')
META_CSV = os.path.join(DATA_DIR, 'metadata.csv')
TOK_DIR = os.path.join(DATA_DIR, 'tokenized')

# ---------- Training config ----------
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 50        # input length
STRIDE = 10         # step between windows (reduce redundancy)
BATCH_SIZE = 256
EPOCHS = 8
LR = 1e-3
GRAD_CLIP = 1.0

EMB_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
DROPOUT = 0.2

TRAIN_SPLIT = 0.9   # split by books
MIN_BOOK_TOKENS = 2000  # skip tiny books

STEPS_PER_EPOCH = 3000
CKPT_DIR = 'checkpoints'
SAVE_EVERY_EPOCH = True
SAVE_EVERY_STEPS = 200
# ------------------------------------

print('DEVICE:', DEVICE)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(os.path.join(CKPT_DIR, 'model2'), exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE: cuda


## Load vocab + metadata

In [2]:
def load_vocab(vocab_path: str) -> Tuple[List[str], Dict[str, int]]:
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(f'vocab.txt not found: {vocab_path}')
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    tok2id = {t: i for i, t in enumerate(vocab)}
    return vocab, tok2id

def read_metadata(meta_csv: str) -> List[dict]:
    if not os.path.exists(meta_csv):
        raise FileNotFoundError(f'metadata.csv not found: {meta_csv}')
    with open(meta_csv, 'r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))

vocab, tok2id = load_vocab(VOCAB_PATH)
rows = read_metadata(META_CSV)

pad_id = tok2id.get('<pad>', 0)
bos_id = tok2id.get('<bos>')
eos_id = tok2id.get('<eos>')

print('vocab size:', len(vocab))
print('special ids:', {'<pad>': pad_id, '<bos>': bos_id, '<eos>': eos_id})
print('metadata rows:', len(rows))

vocab size: 26286
special ids: {'<pad>': 0, '<bos>': 2, '<eos>': 3}
metadata rows: 49


## Dataset (windowed next-token prediction)

In [3]:
def load_ids_file(path: str) -> List[int]:
    with open(path, 'r', encoding='utf-8') as f:
        s = f.read().strip()
    if not s:
        return []
    return list(map(int, s.split()))

@dataclass
class BookData:
    ebook_id: str
    ids_path: str
    token_count: int

def collect_books(rows: List[dict]) -> List[BookData]:
    books = []
    for r in rows:
        ebook_id = r.get('ebook_id')
        if not ebook_id:
            continue
        ids_path = os.path.join(TOK_DIR, f'{ebook_id}.ids.txt')
        if not os.path.exists(ids_path):
            continue
        try:
            token_count = int(r.get('token_count', '0'))
        except Exception:
            token_count = 0
        if token_count < MIN_BOOK_TOKENS:
            continue
        books.append(BookData(str(ebook_id), ids_path, token_count))
    return books

def split_books(books: List[BookData], train_split: float) -> Tuple[List[BookData], List[BookData]]:
    books = books[:]
    random.shuffle(books)
    cut = int(len(books) * train_split)
    return books[:cut], books[cut:]

def load_ids_file(path: str) -> List[int]:
    with open(path, "r", encoding="utf-8") as f:
        s = f.read().strip()
    return list(map(int, s.split())) if s else []

class CachedWindowDataset(Dataset):
    def __init__(self, books, seq_len: int, stride: int):
        self.seq_len = seq_len
        self.stride = stride
        self.book_tokens: List[torch.Tensor] = []
        self.samples: List[Tuple[int, int]] = []

        for b in books:
            ids = load_ids_file(b.ids_path)
            if len(ids) < seq_len + 2:
                continue

            t = torch.tensor(ids, dtype=torch.int32)  # compact in RAM
            bi = len(self.book_tokens)
            self.book_tokens.append(t)

            max_start = t.numel() - (seq_len + 1)
            for st in range(0, max_start + 1, stride):
                self.samples.append((bi, st))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        bi, st = self.samples[idx]
        ids = self.book_tokens[bi]
        chunk = ids[st: st + self.seq_len + 1].to(torch.long)  # convert only this slice
        return chunk[:-1], chunk[1:]

books = collect_books(rows)
if not books:
    raise RuntimeError('No tokenized books found. Check data/tokenized/*.ids.txt and metadata.csv token_count.')

train_books, val_books = split_books(books, TRAIN_SPLIT)
train_ds = CachedWindowDataset(train_books, SEQ_LEN, STRIDE)
val_ds = CachedWindowDataset(val_books, SEQ_LEN, STRIDE)

print(f'Books: total={len(books)} train={len(train_books)} val={len(val_books)}')
print(f'Samples: train={len(train_ds):,} val={len(val_ds):,}')

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,     
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Books: total=49 train=44 val=5
Samples: train=291,495 val=32,338


## Model

In [4]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, num_layers: int, dropout: float, pad_id: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)          # (B,T,E)
        out, _ = self.lstm(e)    # (B,T,H)
        out = self.drop(out)
        logits = self.fc(out)    # (B,T,V)
        return logits

In [ ]:

model = LSTMLM(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    hidden=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_id=pad_id,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

use_amp = (DEVICE == "cuda")
scaler = GradScaler("cuda", enabled=use_amp)

sum(p.numel() for p in model.parameters())

## Sampling utilities

In [5]:
@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]  # (1,V)
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

In [ ]:

prompt = [bos_id] if bos_id is not None else []
print(decode(sample_ids(model, prompt, max_new_tokens=40, temperature=1.0, top_k=50)))

## Train + evaluate

In [6]:
def run_eval(model) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    return avg_loss, ppl

In [ ]:

global_step = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    seen_tokens = 0

    for step, (x, y) in enumerate(train_loader, start=1):
        global_step += 1
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            logits = model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        scaler.scale(loss).backward()

        # if you clip grads, you must unscale first
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(optimizer)   # <-- this is the optimizer step
        scaler.update()

        # logging
        mask = (y != pad_id)
        tok = int(mask.sum().item())
        running += float(loss.item()) * tok
        seen_tokens += tok

        if global_step % SAVE_EVERY_STEPS == 0:
            avg = running / max(1, seen_tokens)
            print(f"epoch {epoch} step {step} train_loss={avg:.4f} train_ppl={math.exp(min(20.0, avg)):.2f}")
            ckpt_path = os.path.join(CKPT_DIR, f"lstm_step{global_step}.pt")
            torch.save({"model_state": model.state_dict(), "vocab": vocab}, ckpt_path)
            print(f"[saved] {ckpt_path}", flush=True)

        if step >= STEPS_PER_EPOCH:
            break

    train_loss = running / max(1, seen_tokens)
    val_loss, val_ppl = run_eval()
    print(f"\n== epoch {epoch} done ==")
    print(f"train_loss={train_loss:.4f} train_ppl={math.exp(min(20.0, train_loss)):.2f}")
    print(f"val_loss={val_loss:.4f}   val_ppl={val_ppl:.2f}")

    # sample (make sure sample_ids sets model.eval() inside, or do it here)
    prompt = [bos_id] if bos_id is not None else []
    out = sample_ids(model, prompt, max_new_tokens=120, temperature=1.0, top_k=50)
    print("\n[sample]")
    print(decode(out))

    if SAVE_EVERY_EPOCH:
        ckpt_path = os.path.join(CKPT_DIR, f"lstm_lm_epoch{epoch}.pt")
        torch.save({
            "model_state": model.state_dict(),
            "vocab": vocab,
            "config": {
                "SEQ_LEN": SEQ_LEN,
                "STRIDE": STRIDE,
                "EMB_DIM": EMB_DIM,
                "HIDDEN_SIZE": HIDDEN_SIZE,
                "NUM_LAYERS": NUM_LAYERS,
                "DROPOUT": DROPOUT,
            }
        }, ckpt_path)
        print(f"[saved] {ckpt_path}\n")

## Tests

In [10]:
def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    ids = [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]
    return ids

In [14]:
prompt = "the older fairies stood all in a group , saying loudly"
prompt_ids = encode_prompt(prompt)
out = sample_ids(model, prompt_ids, temperature=0.9, top_k=50)
print(decode(out))

the older fairies stood all in a group, saying loudly: " my son, i am ready, the youngest knight, who must come to take, then for the sake of the king whom you trusted to me. " then the king ordered his son to take the gardener's daughter to the palace, and there was no more beautiful and beautiful one. she would ask, and the prince said: " you must not be able to come near him. " so she began to look at her eyes. she had hardly ceased to hide her window; and every furrow went round her, and her eyes she was as sweet as ever, and one day she passed through


In [24]:
prompt = "once upon a time"
prompt_ids = encode_prompt(prompt)
out = sample_ids(model, prompt_ids, temperature=0.9, top_k=50)
print(decode(out))

once upon a time the king became the bride of the dead-goddess. " and now that she was a woman of the lake, she would have thought it was too much for her to be as the owner of the box on her own. it was a very pretty baby, and she always was all the fairies. she was always the old woman to herself. what do you suppose, and he is, she would not refuse to help her, since she could not tell her how to make her happy. but the old woman, seeing that she had just left the same time to tell her that she had a beautiful


## Model 2

In [7]:
model2 = LSTMLM(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    hidden=768,
    num_layers=NUM_LAYERS,
    dropout=0.3,
    pad_id=pad_id,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model2.parameters(), lr=LR, weight_decay=1e-4)

use_amp = (DEVICE == "cuda")
scaler = GradScaler("cuda", enabled=use_amp)

sum(p.numel() for p in model2.parameters())

34819758

In [8]:
prompt = [bos_id] if bos_id is not None else []
print(decode(sample_ids(model2, prompt, max_new_tokens=40, temperature=1.0, top_k=50)))

<bos> steer niflheim jerusalem bezestein wisps agog vivid proprietor whichsoever heyday dishes expended pelters chucking tot duga hermione beseem doll's limbed hain't mashonas purse materialism dowered hain't materialism fulfils exhibit agog granny watcher prettier inwardly hog's frothi scots bedraggled reliance ama


In [9]:
global_step = 0

for epoch in range(1, 12 + 1):
    model2.train()
    running = 0.0
    seen_tokens = 0

    for step, (x, y) in enumerate(train_loader, start=1):
        global_step += 1
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            logits = model2(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        scaler.scale(loss).backward()

        # if you clip grads, you must unscale first
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model2.parameters(), GRAD_CLIP)

        scaler.step(optimizer)   # <-- this is the optimizer step
        scaler.update()

        # logging
        mask = (y != pad_id)
        tok = int(mask.sum().item())
        running += float(loss.item()) * tok
        seen_tokens += tok

        if global_step % 500 == 0:
            avg = running / max(1, seen_tokens)
            print(f"epoch {epoch} step {step} train_loss={avg:.4f} train_ppl={math.exp(min(20.0, avg)):.2f}")
            ckpt_path = os.path.join(CKPT_DIR, f"model2/lstm_step{global_step}.pt")
            torch.save({"model_state": model2.state_dict(), "vocab": vocab}, ckpt_path)
            print(f"[saved] {ckpt_path}", flush=True)

        if step >= 5000:
            break

    train_loss = running / max(1, seen_tokens)
    val_loss, val_ppl = run_eval(model2)
    print(f"\n== epoch {epoch} done ==")
    print(f"train_loss={train_loss:.4f} train_ppl={math.exp(min(20.0, train_loss)):.2f}")
    print(f"val_loss={val_loss:.4f}   val_ppl={val_ppl:.2f}")

    # sample (make sure sample_ids sets model.eval() inside, or do it here)
    prompt = [bos_id] if bos_id is not None else []
    out = sample_ids(model2, prompt, max_new_tokens=120, temperature=1.0, top_k=50)
    print("\n[sample]")
    print(decode(out))

    if SAVE_EVERY_EPOCH:
        ckpt_path = os.path.join(CKPT_DIR, f"model2/lstm_lm_epoch{epoch}.pt")
        torch.save({
            "model_state": model2.state_dict(),
            "vocab": vocab,
            "config": {
                "SEQ_LEN": SEQ_LEN,
                "STRIDE": STRIDE,
                "EMB_DIM": EMB_DIM,
                "HIDDEN_SIZE": HIDDEN_SIZE,
                "NUM_LAYERS": NUM_LAYERS,
                "DROPOUT": DROPOUT,
            }
        }, ckpt_path)
        print(f"[saved] {ckpt_path}\n")

epoch 1 step 500 train_loss=5.9974 train_ppl=402.37
[saved] checkpoints\model2/lstm_step500.pt
epoch 1 step 1000 train_loss=5.5986 train_ppl=270.04
[saved] checkpoints\model2/lstm_step1000.pt

== epoch 1 done ==
train_loss=5.5328 train_ppl=252.86
val_loss=4.8825   val_ppl=131.95

[sample]
<bos>, and, we came the mountain to a other of the wood, so to have you to eat me what you have? " " do you be? " he replied. " i have made of you in all this way you; the other has come to the mountain at your hand, " i said. " we has not to get him to look through a time; as a <unk> is one in the forest of the moon comes out or like a white of a <unk> in his feet, and the next horse in him for you as very good-- " she asked him, and that no
[saved] checkpoints\model2/lstm_lm_epoch1.pt

epoch 2 step 362 train_loss=4.9773 train_ppl=145.09
[saved] checkpoints\model2/lstm_step1500.pt
epoch 2 step 862 train_loss=4.9084 train_ppl=135.43
[saved] checkpoints\model2/lstm_step2000.pt

== epoch 2 done ==
train

## Tests 2

In [21]:
prompt = "once upon a time"
prompt_ids = encode_prompt(prompt)
out = sample_ids(model2, prompt_ids, temperature=0.9, top_k=50)
print(decode(out))

once upon a time before he could see the princess, his sister followed them, and told him the best of the king, who was not able to return his daughter. but she would have her leave to go to the princess, and he said, " see i tell you are a little time to be a king. " " oh, my dear! " she cried, " and i hear i am very angry. it is the wicked-headed man. i fear me on my shoulder, but do you hear me that his daughter were so very great, that he would not come to the world. and when he
